# CAT Assignment 1 – ICS1502 Introduction to Machine Learning
**Name:** Lekha Shree  |  **Register No.:** 3122247001031
**Programme:** M.Tech (Integrated) CSE, Semester V, SSN College of Engineering

**Contents**
1. Regression – Mobile Phone Price Prediction (closed form, gradient descent, L2, standardization, λ study, feature importance)
2. Linear classification – Bank Note Authentication (train/test split, L2, accuracy vs λ, 3-D plot, outlier injection)

> **Datasets.** Place `Cellphone.csv` (Mobile Phone Price Prediction) and `BankNote_Authentication.csv` (Bank Note Authentication) in the same folder as this notebook.
> If a file is missing, the notebook falls back to a *synthetic stand-in* so that the code still runs end-to-end. **Final numbers for the report must come from the real files.**

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 160)
SEED = 42
os.makedirs("figures", exist_ok=True)

try:
    display
except NameError:
    display = print

def show(name):
    plt.tight_layout()
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

# ---------- shared helpers ----------
def add_bias(X):
    """Prepend a column of ones -> design matrix with intercept term."""
    return np.c_[np.ones(len(X)), X]

class Standardizer:
    """z = (x - mean_train) / std_train. Statistics come from the TRAINING set only (no leakage)."""
    def fit(self, A):
        self.mu = A.mean(axis=0)
        self.sd = A.std(axis=0)
        self.sd[self.sd == 0] = 1.0
        return self
    def transform(self, A):
        return (A - self.mu) / self.sd

def reg_metrics(y, yhat):
    y, yhat = np.asarray(y).ravel(), np.asarray(yhat).ravel()
    e = y - yhat
    mse = np.mean(e ** 2)
    return {"MSE": mse, "RMSE": np.sqrt(mse), "MAE": np.mean(np.abs(e)),
            "R2": 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2),
            "MAPE_%": np.mean(np.abs(e / y)) * 100}

# Part 1 – Regression: Mobile Phone Price Prediction
## 1.0 Data loading and exploration

In [ ]:
PHONE_CSV, TARGET = "Cellphone.csv", "Price"

if os.path.exists(PHONE_CSV):
    phones = pd.read_csv(PHONE_CSV)
    print("Loaded real dataset:", PHONE_CSV); SYNTH_PHONE = False
else:
    SYNTH_PHONE = True
    print("WARNING: %s not found -> using SYNTHETIC stand-in data (same columns)." % PHONE_CSV)
    r = np.random.default_rng(0); n = 161
    phones = pd.DataFrame({
        "Product_id": np.arange(1, n + 1),
        "Sale": r.integers(10, 3000, n),
        "weight": r.normal(170, 40, n).round(1),
        "resolution": r.normal(5.2, 0.8, n).round(1),
        "ppi": r.normal(430, 120, n).round(0),
        "cpu core": r.choice([2, 4, 6, 8], n),
        "cpu freq": r.normal(1.6, 0.5, n).round(2),
        "internal mem": r.choice([8, 16, 32, 64, 128], n),
        "ram": r.choice([1, 2, 3, 4, 6], n),
        "RearCam": r.normal(10, 4, n).round(1),
        "Front_Cam": r.normal(4, 2, n).round(1),
        "battery": r.normal(2800, 700, n).round(0),
        "thickness": r.normal(8.5, 1.2, n).round(1)})
    phones["Price"] = (-8000 + 2.2 * phones["ppi"] + 2500 * phones["ram"] + 60 * phones["internal mem"]
                       + 1800 * phones["cpu freq"] + 4 * phones["battery"] / 10 + 300 * phones["resolution"]
                       - 3 * phones["weight"] + r.normal(0, 2500, n)).round(0) + 15000

# keep numeric predictors only; drop identifier columns and missing values
phones = phones.drop(columns=[c for c in phones.columns if c.lower() in ("product_id", "id", "unnamed: 0")])
phones = phones.select_dtypes(include=[np.number]).dropna().reset_index(drop=True)
FEATURES = [c for c in phones.columns if c != TARGET]

print("Shape:", phones.shape, "| target:", TARGET, "| #features:", len(FEATURES))
display(phones.head())
display(phones.describe().T)

corr = phones.corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
fig.colorbar(im); ax.set_title("Correlation matrix")
show("reg_corr_heatmap")
print("Correlation of each feature with", TARGET)
print(corr[TARGET].drop(TARGET).sort_values(ascending=False))

## 1.1 Matrix formulation and the effect of the train/test split

**Configuration (justification).**
* Data matrix **X** ∈ ℝ^(n×(d+1)): one row per phone, one column per feature, plus a leading column of ones so that the intercept θ₀ is learned as part of the same parameter vector (ŷ = Xθ).
* Label matrix **y** ∈ ℝ^(n×1): a single column of prices (one real-valued output → regression).
* Parameter matrix **θ** ∈ ℝ^((d+1)×1): one weight per feature plus an intercept. Its size is fixed by the dimension of the data so that Xθ is conformable (n×(d+1))·((d+1)×1) = n×1. Each θⱼ is the change in predicted price for a one-unit change in feature j holding the others fixed, so its sign and magnitude directly show the feature's influence on the prediction.
* Only **training rows** are used to build X and y; test rows are used only for evaluation, which gives an unbiased estimate of generalisation error.

In [ ]:
X_all = phones[FEATURES].values.astype(float)
y_all = phones[[TARGET]].values.astype(float)       # (n, 1)

def closed_form(Xb, y, lam=0.0):
    """theta = (X^T X + lam*I')^-1 X^T y ; I' is identity with 0 for the bias (intercept not penalised)."""
    d = Xb.shape[1]
    R = lam * np.eye(d); R[0, 0] = 0.0
    return np.linalg.pinv(Xb.T @ Xb + R) @ (Xb.T @ y)

# ---- how does the split ratio affect learning? (30 random splits per ratio, plain OLS) ----
rows = []
for ts in [0.1, 0.2, 0.3, 0.4, 0.5]:
    tr_r2, te_r2, te_rmse = [], [], []
    for s in range(30):
        Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=ts, random_state=s)
        th = closed_form(add_bias(Xtr), ytr)
        tr_r2.append(reg_metrics(ytr, add_bias(Xtr) @ th)["R2"])
        m = reg_metrics(yte, add_bias(Xte) @ th); te_r2.append(m["R2"]); te_rmse.append(m["RMSE"])
    rows.append({"test_size": ts, "train_R2": np.mean(tr_r2), "test_R2_mean": np.mean(te_r2),
                 "test_R2_std": np.std(te_r2), "test_RMSE_mean": np.mean(te_rmse)})
split_df = pd.DataFrame(rows); display(split_df)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].errorbar(split_df.test_size, split_df.test_R2_mean, yerr=split_df.test_R2_std, marker="o", capsize=3, label="test R²")
ax[0].plot(split_df.test_size, split_df.train_R2, marker="s", label="train R²")
ax[0].set_xlabel("test fraction"); ax[0].set_ylabel("R²"); ax[0].legend(); ax[0].set_title("Effect of split ratio")
ax[1].plot(split_df.test_size, split_df.test_RMSE_mean, marker="o"); ax[1].set_xlabel("test fraction")
ax[1].set_ylabel("test RMSE"); ax[1].set_title("Test RMSE vs split")
show("reg_split_effect")

# ---- final split used for the rest of the assignment: 80 / 20 ----
Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, random_state=SEED)
Xtr_b, Xte_b = add_bias(Xtr), add_bias(Xte)
print("Data matrix  X_train :", Xtr_b.shape, "(n_train x (d+1))")
print("Label matrix y_train :", ytr.shape)
print("Parameter    theta   :", (Xtr_b.shape[1], 1))
print("Test matrices        :", Xte_b.shape, yte.shape)
print("cond(X^T X) raw          : %.3e" % np.linalg.cond(Xtr_b.T @ Xtr_b))
sx_tmp = Standardizer().fit(Xtr)
print("cond(X^T X) standardized : %.3e" % np.linalg.cond(add_bias(sx_tmp.transform(Xtr)).T @ add_bias(sx_tmp.transform(Xtr))))

**Reading the split study.** A small test set gives a noisy (high-variance) estimate of test error, while a large test set leaves too little data for training, so the learned θ is poorer. The 80/20 split is a standard compromise and is used below.

## 1.2 (Q1) Linear regression using the closed-form (normal-equation) solution

In [ ]:
theta_cf = closed_form(Xtr_b, ytr)
pred_cf_tr, pred_cf_te = Xtr_b @ theta_cf, Xte_b @ theta_cf

theta_df = pd.DataFrame({"parameter": ["intercept"] + FEATURES, "theta (raw units)": theta_cf.ravel()})
display(theta_df)
res_cf = {"train": reg_metrics(ytr, pred_cf_tr), "test": reg_metrics(yte, pred_cf_te)}
display(pd.DataFrame(res_cf).T)

## 1.3 (Q2) Linear regression using gradient descent

Gradient descent minimises J(θ) = (1/2n)‖Xθ − y‖² + (λ/2n)‖θ₁:ₙ‖² with update θ ← θ − α∇J.
Raw features have wildly different scales (e.g. battery ≈ 10³, CPU frequency ≈ 1), which makes the loss surface a long narrow valley, so GD is run on **standardized features and a standardized target** (statistics from the training set only) and the predictions are converted back to rupee/price units.
The closed form is scale-invariant for λ = 0, so the two solutions can be compared directly.

In [ ]:
def gradient_descent(Xb, y, lam=0.0, lr=0.05, epochs=20000, tol=1e-13):
    n, d = Xb.shape
    theta = np.zeros((d, 1)); hist = []
    lr = min(lr, 1.0 / (np.linalg.eigvalsh(Xb.T @ Xb / n).max() + lam / n))   # keep the step size stable for any λ
    for ep in range(epochs):
        resid = Xb @ theta - y
        grad = Xb.T @ resid / n
        pen = (lam / n) * theta; pen[0] = 0.0          # bias is not penalised
        theta = theta - lr * (grad + pen)
        hist.append(0.5 * np.mean(resid ** 2) + lam / (2 * n) * np.sum(theta[1:] ** 2))
        if not np.isfinite(hist[-1]): break
        if ep > 0 and abs(hist[-2] - hist[-1]) < tol: break
    return theta, np.array(hist)

def fit_predict_std(Xa, ya, Xb_, lam=0.0, method="closed", lr=0.05, epochs=20000):
    """Standardize X and y with train statistics, fit, return predictions in ORIGINAL units."""
    sx = Standardizer().fit(Xa); ym, ys = ya.mean(), ya.std()
    Za, Zb = add_bias(sx.transform(Xa)), add_bias(sx.transform(Xb_))
    t = (ya - ym) / ys
    if method == "closed":
        theta, hist = closed_form(Za, t, lam), None
    else:
        theta, hist = gradient_descent(Za, t, lam, lr, epochs)
    return {"theta": theta, "hist": hist, "pred_tr": (Za @ theta) * ys + ym, "pred_te": (Zb @ theta) * ys + ym}

gd = fit_predict_std(Xtr, ytr, Xte, lam=0.0, method="gd")
cf_std = fit_predict_std(Xtr, ytr, Xte, lam=0.0, method="closed")
print("GD stopped after %d iterations, final loss %.6f" % (len(gd["hist"]), gd["hist"][-1]))
print("Max |prediction difference| closed form vs GD on test set: %.4f" % np.max(np.abs(gd["pred_te"] - pred_cf_te)))

res_q12 = pd.DataFrame({"Closed form (train)": reg_metrics(ytr, pred_cf_tr), "Closed form (test)": reg_metrics(yte, pred_cf_te),
                        "GD (train)": reg_metrics(ytr, gd["pred_tr"]), "GD (test)": reg_metrics(yte, gd["pred_te"])}).T
display(res_q12)

plt.figure(figsize=(5, 3.5)); plt.plot(gd["hist"]); plt.yscale("log")
plt.xlabel("iteration"); plt.ylabel("loss J(θ)"); plt.title("Gradient-descent convergence")
show("reg_gd_convergence")

## 1.4 (Q3) Predicted vs. actual on the test data – closed form and gradient descent

In [ ]:
def pred_vs_actual(ax, y, yhat, title):
    y, yhat = y.ravel(), yhat.ravel()
    ax.scatter(y, yhat, s=22, alpha=0.75)
    lo, hi = min(y.min(), yhat.min()), max(y.max(), yhat.max())
    ax.plot([lo, hi], [lo, hi], "r--", label="ideal (y = ŷ)")
    ax.set_xlabel("actual price"); ax.set_ylabel("predicted price")
    ax.set_title(f"{title}\nR²={reg_metrics(y, yhat)['R2']:.3f}"); ax.legend()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
pred_vs_actual(ax[0], yte, pred_cf_te, "Closed form (test)")
pred_vs_actual(ax[1], yte, gd["pred_te"], "Gradient descent (test)")
show("reg_q3_pred_vs_actual")

# residual (error) analysis
res = yte.ravel() - pred_cf_te.ravel()
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].scatter(pred_cf_te, res, s=20); ax[0].axhline(0, color="r", ls="--"); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("residual"); ax[0].set_title("Residuals vs predicted")
ax[1].hist(res, bins=15, edgecolor="k"); ax[1].set_title("Residual distribution"); ax[1].set_xlabel("residual")
show("reg_q3_residuals")

## 1.5 (Q4) Repeat Q1–Q3 with L2 (ridge) regularization
Ridge closed form: θ = (XᵀX + λI′)⁻¹Xᵀy (I′ = identity with a 0 for the intercept). The same penalty is added to the gradient for GD.
λ is chosen by **5-fold cross-validation on the training set only**; the test set is never used for tuning.

In [ ]:
def cv_lambda(Xa, ya, lams, k=5):
    idx = np.random.default_rng(0).permutation(len(Xa)); folds = np.array_split(idx, k); out = []
    for lam in lams:
        errs = []
        for f in folds:
            tr = np.setdiff1d(idx, f)
            p = fit_predict_std(Xa[tr], ya[tr], Xa[f], lam)["pred_te"]
            errs.append(np.mean((ya[f] - p) ** 2))
        out.append(np.mean(errs))
    return np.array(out)

lam_grid = np.logspace(-3, 3, 13)
cv_mse = cv_lambda(Xtr, ytr, lam_grid)
LAM_BEST = lam_grid[np.argmin(cv_mse)]
print("Best λ by 5-fold CV: %.4g" % LAM_BEST)
plt.figure(figsize=(5, 3.5)); plt.semilogx(lam_grid, cv_mse, "o-"); plt.axvline(LAM_BEST, color="r", ls="--")
plt.xlabel("λ"); plt.ylabel("CV MSE"); plt.title("λ selection"); show("reg_lambda_cv")

rid_cf = fit_predict_std(Xtr, ytr, Xte, LAM_BEST, "closed")
rid_gd = fit_predict_std(Xtr, ytr, Xte, LAM_BEST, "gd")
print("max |closed - GD| on test predictions: %.4f" % np.max(np.abs(rid_cf["pred_te"] - rid_gd["pred_te"])))

res_q4 = pd.DataFrame({
    "OLS closed (test)": reg_metrics(yte, pred_cf_te), "OLS GD (test)": reg_metrics(yte, gd["pred_te"]),
    "Ridge closed (test)": reg_metrics(yte, rid_cf["pred_te"]), "Ridge GD (test)": reg_metrics(yte, rid_gd["pred_te"]),
    "Ridge closed (train)": reg_metrics(ytr, rid_cf["pred_tr"]), "Ridge GD (train)": reg_metrics(ytr, rid_gd["pred_tr"])}).T
display(res_q4)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
pred_vs_actual(ax[0], yte, rid_cf["pred_te"], f"Ridge closed form, λ={LAM_BEST:.3g} (test)")
pred_vs_actual(ax[1], yte, rid_gd["pred_te"], f"Ridge gradient descent, λ={LAM_BEST:.3g} (test)")
show("reg_q4_ridge_pred_vs_actual")

## 1.6 (Q5) Ridge with and without data standardization
With raw features the L2 penalty treats all weights equally even though their units differ: a feature measured in large numbers (battery) needs a *small* weight and is barely penalised, whereas a small-valued feature (CPU frequency) needs a *large* weight and is penalised heavily. Standardization puts all features on the same scale so the penalty is fair, and it also fixes the ill-conditioning that makes plain gradient descent crawl.

In [ ]:
def gd_raw(Xa, ya, Xb_, lam, epochs=20000):
    Xab, Xbb = add_bias(Xa), add_bias(Xb_)
    L = np.linalg.eigvalsh(Xab.T @ Xab / len(Xab)).max()      # largest curvature -> biggest safe step
    th, hist = gradient_descent(Xab, ya, lam, lr=1.0 / L, epochs=epochs, tol=0)
    return {"theta": th, "hist": hist, "pred_te": Xbb @ th, "pred_tr": Xab @ th}

raw_cf_pred = add_bias(Xte) @ closed_form(add_bias(Xtr), ytr, LAM_BEST)
raw_cf_tr   = add_bias(Xtr) @ closed_form(add_bias(Xtr), ytr, LAM_BEST)
raw_gd = gd_raw(Xtr, ytr, Xte, LAM_BEST)

res_q5 = pd.DataFrame({
    "Ridge closed form – RAW (test)": reg_metrics(yte, raw_cf_pred),
    "Ridge closed form – STANDARDIZED (test)": reg_metrics(yte, rid_cf["pred_te"]),
    "Ridge GD – RAW (test)": reg_metrics(yte, raw_gd["pred_te"]),
    "Ridge GD – STANDARDIZED (test)": reg_metrics(yte, rid_gd["pred_te"])}).T
display(res_q5)
print("GD iterations -> raw: %d (final loss %.4g) | standardized: %d (final loss %.4g)"
      % (len(raw_gd["hist"]), raw_gd["hist"][-1], len(rid_gd["hist"]), rid_gd["hist"][-1]))

# same λ means different amounts of shrinkage in the two settings -> compare across λ
lams5 = np.logspace(-2, 4, 13); r_raw, r_std = [], []
for lam in lams5:
    r_raw.append(reg_metrics(yte, add_bias(Xte) @ closed_form(add_bias(Xtr), ytr, lam))["R2"])
    r_std.append(reg_metrics(yte, fit_predict_std(Xtr, ytr, Xte, lam)["pred_te"])["R2"])
plt.figure(figsize=(6, 3.8)); plt.semilogx(lams5, r_raw, "o-", label="raw features"); plt.semilogx(lams5, r_std, "s-", label="standardized")
plt.xlabel("λ"); plt.ylabel("test R²"); plt.legend(); plt.title("Ridge: raw vs standardized"); show("reg_q5_raw_vs_std")

## 1.7 (Q6) Predicted vs. actual on the test data for different λ

In [ ]:
LAMS6 = [0, 0.1, 1, 10, 100, 1000, 10000, 100000]
fig, axes = plt.subplots(2, 4, figsize=(17, 8)); rows = []
for ax, lam in zip(axes.ravel(), LAMS6):
    out = fit_predict_std(Xtr, ytr, Xte, lam)
    pred_vs_actual(ax, yte, out["pred_te"], f"λ = {lam:g}")
    m = reg_metrics(yte, out["pred_te"]); rows.append({"lambda": lam, "test_R2": m["R2"], "test_RMSE": m["RMSE"],
                                                       "train_R2": reg_metrics(ytr, out["pred_tr"])["R2"],
                                                       "||theta||": np.linalg.norm(out["theta"][1:])})
show("reg_q6_lambda_grid"); lam_df = pd.DataFrame(rows); display(lam_df)

**Interpretation.** λ = 0 is ordinary least squares (lowest bias, highest variance). A moderate λ shrinks the weights, trading a little bias for lower variance, and can improve test error when features are correlated. A very large λ forces all weights to ≈ 0, so every prediction collapses towards the mean price (points form a flat horizontal line in the plot → under-fitting).

## 1.8 (Q7) Feature importance from the L2 weights

In [ ]:
w_ridge = rid_cf["theta"][1:].ravel(); w_ols = cf_std["theta"][1:].ravel()
imp = pd.DataFrame({"feature": FEATURES, "ridge_weight": w_ridge, "|ridge_weight|": np.abs(w_ridge), "OLS_weight": w_ols}
                   ).sort_values("|ridge_weight|", ascending=False).reset_index(drop=True)
imp["rank"] = imp.index + 1; display(imp)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
o = imp.iloc[::-1]
ax[0].barh(o.feature, o.ridge_weight, color=["tab:green" if v > 0 else "tab:red" for v in o.ridge_weight])
ax[0].axvline(0, color="k", lw=0.8); ax[0].set_title(f"Standardized ridge weights (λ={LAM_BEST:.3g})"); ax[0].set_xlabel("weight (per 1 σ of feature)")
paths = np.array([fit_predict_std(Xtr, ytr, Xte, l)["theta"][1:].ravel() for l in np.logspace(-2, 4, 40)])
for j, f in enumerate(FEATURES): ax[1].semilogx(np.logspace(-2, 4, 40), paths[:, j], label=f)
ax[1].set_xlabel("λ"); ax[1].set_ylabel("weight"); ax[1].set_title("Ridge coefficient paths"); ax[1].legend(fontsize=6, ncol=2)
show("reg_q7_feature_importance")
print("Top 3 features:", list(imp.feature[:3]))

**Why standardized weights?** After standardization each weight is the price change for a 1-σ change in that feature, so magnitudes are directly comparable. Ridge shrinks weights of weak or collinear features towards 0 while strong predictors survive, so the ranking by |w| is a stable importance measure (positive = raises price, negative = lowers it). Correlated features share weight, so a low weight does not always mean "irrelevant" – check the correlation matrix too.

---
# Part 2 – Linear Classification: Bank Note Authentication
**Model:** a single-layer neural network (logistic regression): p(y=1|x) = σ(wᵀx + b), trained by gradient descent on the binary cross-entropy loss with optional L2 penalty (λ/2n)‖w‖². Logistic regression is appropriate because the task is 2-class, the four wavelet features are numeric, and the classes are close to linearly separable.

In [ ]:
BANK_CSV = "BankNote_Authentication.csv"
if os.path.exists(BANK_CSV):
    bank = pd.read_csv(BANK_CSV); print("Loaded real dataset:", BANK_CSV); SYNTH_BANK = False
else:
    SYNTH_BANK = True
    print("WARNING: %s not found -> using SYNTHETIC stand-in data (same columns)." % BANK_CSV)
    r = np.random.default_rng(1)
    g = np.c_[r.normal(2.3, 2.0, 762), r.normal(4.3, 5.1, 762), r.normal(0.8, 3.2, 762), r.normal(-1.15, 2.1, 762), np.zeros(762)]
    f = np.c_[r.normal(-1.9, 1.9, 610), r.normal(-1.0, 5.4, 610), r.normal(2.1, 5.3, 610), r.normal(-1.25, 2.1, 610), np.ones(610)]
    bank = pd.DataFrame(np.r_[g, f], columns=["variance", "skewness", "curtosis", "entropy", "class"])
bank.columns = [c.strip().lower() for c in bank.columns]
bank = bank.dropna().reset_index(drop=True)
CLS_FEATURES = [c for c in bank.columns if c != "class"]
print(bank.shape); display(bank.head()); display(bank.describe().T)
print("Class balance:\n", bank["class"].value_counts())

fig, ax = plt.subplots(1, 4, figsize=(15, 3))
for a, c in zip(ax, CLS_FEATURES):
    for k in (0, 1): a.hist(bank.loc[bank["class"] == k, c], bins=25, alpha=0.6, label=f"class {k}")
    a.set_title(c)
ax[0].legend(); show("clf_feature_hists")

## 2.1 (Q1) Train / test split

In [ ]:
Xc = bank[CLS_FEATURES].values.astype(float); yc = bank["class"].values.astype(float).reshape(-1, 1)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=SEED, stratify=yc)   # stratified keeps class ratio
print("train:", Xc_tr.shape, " test:", Xc_te.shape)
print("class-1 fraction  train=%.3f  test=%.3f" % (yc_tr.mean(), yc_te.mean()))

scx = Standardizer().fit(Xc_tr)
Ztr, Zte = add_bias(scx.transform(Xc_tr)), add_bias(scx.transform(Xc_te))

def sigmoid(z): return np.where(z >= 0, 1 / (1 + np.exp(-np.abs(z))), np.exp(-np.abs(z)) / (1 + np.exp(-np.abs(z))))

def fit_logreg(Zb, y, lam=0.0, lr=0.5, epochs=5000):
    n, d = Zb.shape; w = np.zeros((d, 1)); hist = []
    lr = min(lr, 1.0 / (0.25 * np.linalg.eigvalsh(Zb.T @ Zb / n).max() + lam / n))   # stable step for any λ
    for _ in range(epochs):
        p = sigmoid(Zb @ w)
        pen = (lam / n) * w; pen[0] = 0
        w = w - lr * (Zb.T @ (p - y) / n + pen)
        pc = np.clip(p, 1e-12, 1 - 1e-12)
        hist.append(-np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc)) + lam / (2 * n) * np.sum(w[1:] ** 2))
    return w, np.array(hist)

def predict(Zb, w): return (sigmoid(Zb @ w) >= 0.5).astype(float)
def acc(Zb, y, w): return float(np.mean(predict(Zb, w) == y))
def clf_report(Zb, y, w):
    p = predict(Zb, w); tp = np.sum((p == 1) & (y == 1)); fp = np.sum((p == 1) & (y == 0)); fn = np.sum((p == 0) & (y == 1)); tn = np.sum((p == 0) & (y == 0))
    pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
    return {"accuracy": (tp + tn) / len(y), "precision": pr, "recall": rc, "F1": 2 * pr * rc / max(pr + rc, 1e-12), "TN": tn, "FP": fp, "FN": fn, "TP": tp}

## 2.2 (Q2) Classifier with and without L2 regularization

In [ ]:
# λ chosen by 5-fold CV on the training set (largest λ within tolerance of the best CV accuracy)
def cv_logreg(Xa, ya, lams, k=5, epochs=2000):
    idx = np.random.default_rng(0).permutation(len(Xa)); folds = np.array_split(idx, k); out = []
    for lam in lams:
        a = []
        for f in folds:
            tr = np.setdiff1d(idx, f); s = Standardizer().fit(Xa[tr])
            w, _ = fit_logreg(add_bias(s.transform(Xa[tr])), ya[tr], lam, epochs=epochs)
            a.append(acc(add_bias(s.transform(Xa[f])), ya[f], w))
        out.append(np.mean(a))
    return np.array(out)

lam_c = np.logspace(-2, 3, 11)
cv_acc = cv_logreg(Xc_tr, yc_tr, lam_c)
LAM_CLF = lam_c[np.where(cv_acc >= cv_acc.max() - 1e-9)[0].max()]
print("CV accuracies:", dict(zip(np.round(lam_c, 3), np.round(cv_acc, 4)))); print("Chosen λ =", LAM_CLF)

w0, h0 = fit_logreg(Ztr, yc_tr, 0.0)
w1, h1 = fit_logreg(Ztr, yc_tr, LAM_CLF)
tab = pd.DataFrame({
    "No L2 (λ=0) – train": clf_report(Ztr, yc_tr, w0), "No L2 (λ=0) – test": clf_report(Zte, yc_te, w0),
    f"L2 (λ={LAM_CLF:g}) – train": clf_report(Ztr, yc_tr, w1), f"L2 (λ={LAM_CLF:g}) – test": clf_report(Zte, yc_te, w1)}).T
display(tab)
print("||w|| without L2 = %.3f | with L2 = %.3f" % (np.linalg.norm(w0[1:]), np.linalg.norm(w1[1:])))

from sklearn.linear_model import LogisticRegression    # sanity check only
sk = LogisticRegression(C=1e6, max_iter=5000).fit(Ztr[:, 1:], yc_tr.ravel())
print("sklearn sanity-check test accuracy (no penalty): %.4f" % sk.score(Zte[:, 1:], yc_te.ravel()))

plt.figure(figsize=(5.5, 3.5)); plt.plot(h0, label="λ=0"); plt.plot(h1, label=f"λ={LAM_CLF:g}")
plt.xlabel("iteration"); plt.ylabel("loss"); plt.yscale("log"); plt.legend(); plt.title("Training loss"); show("clf_loss_curves")

## 2.3 (Q3) Training and test accuracy vs. λ

In [ ]:
lams = np.logspace(-3, 4, 22); tr_a, te_a = [], []
for lam in lams:
    w, _ = fit_logreg(Ztr, yc_tr, lam, epochs=3000); tr_a.append(acc(Ztr, yc_tr, w)); te_a.append(acc(Zte, yc_te, w))
plt.figure(figsize=(7, 4))
plt.semilogx(lams, tr_a, "o-", label="train accuracy"); plt.semilogx(lams, te_a, "s-", label="test accuracy")
plt.axhline(acc(Zte, yc_te, w0), color="gray", ls=":", label="test acc, λ=0 (no L2)")
plt.xlabel("λ"); plt.ylabel("accuracy"); plt.title("Accuracy vs λ"); plt.legend(); plt.grid(alpha=0.3); show("clf_acc_vs_lambda")
display(pd.DataFrame({"lambda": lams, "train_acc": tr_a, "test_acc": te_a}))

## 2.4 (Q4) 3-D visualization using the three most important features

In [ ]:
imp_c = pd.Series(np.abs(w1[1:].ravel()), index=CLS_FEATURES).sort_values(ascending=False)
print("Feature importance |w| (standardized, L2 model):"); print(imp_c)
TOP3 = list(imp_c.index[:3]); idx3 = [CLS_FEATURES.index(f) for f in TOP3]

# classifier restricted to the 3 features so we can draw its decision plane
s3 = Standardizer().fit(Xc_tr[:, idx3])
Z3tr, Z3te = add_bias(s3.transform(Xc_tr[:, idx3])), add_bias(s3.transform(Xc_te[:, idx3]))
w3, _ = fit_logreg(Z3tr, yc_tr, LAM_CLF)
acc3 = acc(Z3te, yc_te, w3)
print("3-feature model test accuracy: %.4f" % acc3)

def plot3d(Zp, yp, w, title, name, outlier_mask=None, views=((20, 45), (20, 135))):
    fig = plt.figure(figsize=(13, 5.5))
    # solve the plane w0 + w1*x + w2*y + w3*z = 0 for the axis with the LARGEST |weight| (well-conditioned plane)
    k = int(np.argmax(np.abs(w[1:4, 0]))) + 1; o1, o2 = [j for j in (1, 2, 3) if j != k]
    g1, g2 = np.meshgrid(np.linspace(Zp[:, o1].min(), Zp[:, o1].max(), 20), np.linspace(Zp[:, o2].min(), Zp[:, o2].max(), 20))
    gk = -(w[0, 0] + w[o1, 0] * g1 + w[o2, 0] * g2) / w[k, 0]
    gk = np.where((gk >= Zp[:, k].min()) & (gk <= Zp[:, k].max()), gk, np.nan)
    P = {o1: g1, o2: g2, k: gk}
    for i, (el, az) in enumerate(views):
        ax = fig.add_subplot(1, 2, i + 1, projection="3d")
        for kk, col in ((0, "tab:blue"), (1, "tab:orange")):
            m = (yp.ravel() == kk)
            ax.scatter(Zp[m, 1], Zp[m, 2], Zp[m, 3], c=col, s=10, alpha=0.6, label=f"class {kk}")
        if outlier_mask is not None:
            ax.scatter(Zp[outlier_mask, 1], Zp[outlier_mask, 2], Zp[outlier_mask, 3], c="red", marker="x", s=45, label="shifted outliers")
        ax.plot_surface(P[1], P[2], P[3], alpha=0.3, color="gray")
        ax.set_xlim(Zp[:, 1].min(), Zp[:, 1].max()); ax.set_ylim(Zp[:, 2].min(), Zp[:, 2].max()); ax.set_zlim(Zp[:, 3].min(), Zp[:, 3].max())
        ax.set_xlabel(TOP3[0] + " (z)"); ax.set_ylabel(TOP3[1] + " (z)"); ax.set_zlabel(TOP3[2] + " (z)", labelpad=8)
        ax.view_init(el, az); ax.set_title(title, fontsize=9)
    ax.legend(loc="upper left", fontsize=7); show(name)

plot3d(Z3te, yc_te, w3, "Test set with learned decision plane", "clf_q4_3d")

## 2.5 (Q5) Intentionally introduce outliers by shifting data points
A fraction of the **training** points is moved far from the data cloud: each selected point is shifted by ±6 standard deviations along the three most important features (random sign per point). Labels are left unchanged, so these are "extreme-value" outliers. The standardizer is re-fitted on the contaminated training data (as would happen in practice), and the model is always evaluated on the **clean test set**.

In [ ]:
def inject_outliers(X, frac, k_sigma=6.0, cols=idx3, seed=7):
    r = np.random.default_rng(seed); Xo = X.copy(); n_out = int(round(frac * len(X)))
    chosen = r.choice(len(X), n_out, replace=False) if n_out else np.array([], dtype=int)
    sd = X.std(axis=0)
    for c in cols: Xo[chosen, c] += r.choice([-1, 1], n_out) * k_sigma * sd[c]
    mask = np.zeros(len(X), bool); mask[chosen] = True
    return Xo, mask

def fit_eval(Xa, ya, lam, epochs=5000):
    s = Standardizer().fit(Xa); Za = add_bias(s.transform(Xa))
    w, _ = fit_logreg(Za, ya, lam, epochs=epochs)
    return w, s, acc(Za, ya, w), acc(add_bias(s.transform(Xc_te)), yc_te, w)   # test set stays clean

Xc_out, out_mask = inject_outliers(Xc_tr, 0.05)
print("Injected %d outliers out of %d training samples" % (out_mask.sum(), len(Xc_tr)))
display(pd.DataFrame({"feature": CLS_FEATURES, "mean (clean)": Xc_tr.mean(0), "mean (outliers)": Xc_out.mean(0),
                      "std (clean)": Xc_tr.std(0), "std (outliers)": Xc_out.std(0)}))

# visualise clean vs contaminated training data in the original feature space
fig = plt.figure(figsize=(12, 5))
for i, (A, ttl) in enumerate(((Xc_tr, "Clean training data"), (Xc_out, "After shifting outliers"))):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    for k, col in ((0, "tab:blue"), (1, "tab:orange")):
        m = (yc_tr.ravel() == k) & ~(out_mask if i else np.zeros_like(out_mask)); ax.scatter(A[m, idx3[0]], A[m, idx3[1]], A[m, idx3[2]], c=col, s=8, alpha=0.5)
    if i: ax.scatter(A[out_mask, idx3[0]], A[out_mask, idx3[1]], A[out_mask, idx3[2]], c="red", marker="x", s=40, label="outliers"); ax.legend()
    ax.set_xlabel(TOP3[0]); ax.set_ylabel(TOP3[1]); ax.set_zlabel(TOP3[2]); ax.set_title(ttl)
show("clf_q5_outliers_3d")

## 2.6 (Q6) Fit the classifier on outlier-injected data and comment on the impact

In [ ]:
rows = []
for name, A in (("clean", Xc_tr), ("5% outliers", Xc_out)):
    for lam in (0.0, LAM_CLF):
        w, s, a_tr, a_te = fit_eval(A, yc_tr, lam)
        rows.append({"training data": name, "λ": lam, "train_acc": a_tr, "clean_test_acc": a_te, "||w||": np.linalg.norm(w[1:])})
cmp_df = pd.DataFrame(rows); display(cmp_df)

# sweep contamination level
fracs = [0, 0.01, 0.02, 0.05, 0.10, 0.20]; sweep = {0.0: [], LAM_CLF: []}
for fr in fracs:
    Xo, _ = inject_outliers(Xc_tr, fr)
    for lam in sweep: sweep[lam].append(fit_eval(Xo, yc_tr, lam)[3])
plt.figure(figsize=(6.5, 4))
plt.plot([f * 100 for f in fracs], sweep[0.0], "o-", label="no L2"); plt.plot([f * 100 for f in fracs], sweep[LAM_CLF], "s-", label=f"L2 λ={LAM_CLF:g}")
plt.xlabel("% of training points shifted"); plt.ylabel("clean-test accuracy"); plt.title("Impact of outliers"); plt.legend(); plt.grid(alpha=0.3); show("clf_q6_outlier_sweep")

# decision boundary comparison
w_clean, s_clean, _, _ = fit_eval(Xc_tr, yc_tr, LAM_CLF); w_out, s_out, _, _ = fit_eval(Xc_out, yc_tr, LAM_CLF)
print("Weights (clean)   :", w_clean.ravel()); print("Weights (outliers):", w_out.ravel())
cos_sim = float(w_clean[1:].ravel() @ w_out[1:].ravel()) / (np.linalg.norm(w_clean[1:]) * np.linalg.norm(w_out[1:]))
print("Cosine similarity of weight vectors: %.3f" % cos_sim)

drop0 = cmp_df.query("λ == 0").clean_test_acc.iloc[0] - cmp_df.query("λ == 0").clean_test_acc.iloc[1]
dropL = cmp_df.query("λ != 0").clean_test_acc.iloc[0] - cmp_df.query("λ != 0").clean_test_acc.iloc[1]
print("\nAccuracy drop caused by outliers: no-L2 = %.4f, L2 = %.4f" % (drop0, dropL))
print("Comment:", "the un-regularized model is noticeably affected by the shifted points" if drop0 > 0.01 else "the model is fairly robust at this contamination level",
      "; L2 regularization", "reduced" if dropL < drop0 else "did not reduce", "the damage.")

**Commentary template (edit using your own numbers).** Logistic regression minimises a loss that grows with distance from the boundary for misclassified points, so a few extreme points can drag the boundary and inflate the weights (look at ‖w‖ and the cosine similarity of the weight vectors). Standardization is also distorted because the outliers inflate the mean and σ. L2 regularization limits weight growth and typically reduces, but does not remove, this sensitivity; robust remedies are outlier removal, robust scaling (median/IQR), winsorizing, or a robust loss.

---
## Summary / Conclusions
* **Regression:** the matrix (normal-equation) approach and gradient descent give essentially identical predictions once features are standardized; ridge helps with correlated features; standardization is essential for fair L2 penalties and for GD convergence. Feature importance comes from standardized ridge weights.
* **Classification:** a single-layer logistic model separates genuine from forged notes very well; L2 shrinks weights with little change in accuracy; accuracy vs λ shows the under-fitting regime at large λ; outliers degrade the boundary, and regularization only partly mitigates this.

---
# Part 3 – Auto-generate the final report (PDF)
The next cell collects every table, number and figure produced above and writes **`CAT1_Report_Lekha_Shree_3122247001031.pdf`** with data-driven commentary. Run the notebook top to bottom on the real datasets, then run this cell last.

In [ ]:
import sys, subprocess, datetime, matplotlib
try:
    import reportlab
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "reportlab"])
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak, KeepTogether
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.utils import ImageReader

fd = os.path.join(matplotlib.get_data_path(), "fonts", "ttf")
pdfmetrics.registerFont(TTFont("DV", os.path.join(fd, "DejaVuSans.ttf")))
pdfmetrics.registerFont(TTFont("DV-B", os.path.join(fd, "DejaVuSans-Bold.ttf")))
pdfmetrics.registerFont(TTFont("DV-I", os.path.join(fd, "DejaVuSans-Oblique.ttf")))
pdfmetrics.registerFontFamily("DV", normal="DV", bold="DV-B", italic="DV-I", boldItalic="DV-B")

ss = getSampleStyleSheet()
B = ParagraphStyle("B", parent=ss["BodyText"], fontName="DV", fontSize=9.5, leading=13.5, spaceAfter=5)
H1 = ParagraphStyle("H1", parent=B, fontName="DV-B", fontSize=14, leading=18, spaceBefore=10, spaceAfter=6, textColor=colors.HexColor("#1F3864"))
H2 = ParagraphStyle("H2", parent=B, fontName="DV-B", fontSize=11, leading=15, spaceBefore=8, spaceAfter=4, textColor=colors.HexColor("#2F5496"))
CAP = ParagraphStyle("CAP", parent=B, fontName="DV-I", fontSize=8, alignment=1, textColor=colors.grey)
TC = ParagraphStyle("TC", parent=B, fontSize=7.5, leading=9.5, spaceAfter=0)
TCB = ParagraphStyle("TCB", parent=TC, fontName="DV-B")
BUL = ParagraphStyle("BUL", parent=B, leftIndent=14, bulletIndent=2)

def fmt(v):
    if isinstance(v, (int, np.integer)): return f"{v}"
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v): return "nan"
        return f"{v:,.0f}" if abs(v) >= 1000 else f"{v:.4g}" if abs(v) < 1e-3 and v != 0 else f"{v:.4f}"
    return str(v)

def tbl(df, index=True, col_w=None):
    d = df.reset_index() if index else df
    data = [[Paragraph(str(c), TCB) for c in d.columns]] + [[Paragraph(fmt(v), TC) for v in r] for r in d.values.tolist()]
    t = Table(data, colWidths=col_w, repeatRows=1)
    t.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.4, colors.grey), ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9E2F3")),
                           ("VALIGN", (0, 0), (-1, -1), "MIDDLE"), ("TOPPADDING", (0, 0), (-1, -1), 2), ("BOTTOMPADDING", (0, 0), (-1, -1), 2)]))
    return t

def fig(name, caption, width=15.5):
    p = f"figures/{name}.png"
    if not os.path.exists(p): return [Paragraph(f"[missing figure {name}]", B)]
    w, h = ImageReader(p).getSize()
    return [Image(p, width=width * cm, height=width * cm * h / w), Paragraph(caption, CAP), Spacer(1, 6)]

def P(t): return Paragraph(t, B)
def bullets(items): return [Paragraph(i, BUL, bulletText="•") for i in items]

S = []   # story
SYN = SYNTH_PHONE or SYNTH_BANK
if SYN:
    S.append(Paragraph("<font color='red'><b>PREVIEW ONLY – generated from SYNTHETIC stand-in data (dataset CSV not found). Do not submit; re-run with the real datasets.</b></font>", B))

# ---------------- title ----------------
S += [Paragraph("Sri Sivasubramaniya Nadar College of Engineering, Chennai", ParagraphStyle("t1", parent=H1, alignment=1, fontSize=15)),
      Paragraph("Department of Computer Science and Engineering", ParagraphStyle("t2", parent=B, alignment=1)),
      Paragraph("ICS1502 – Introduction to Machine Learning<br/>CAT Assignment 1 – Report", ParagraphStyle("t3", parent=H1, alignment=1, fontSize=17, leading=22)),
      tbl(pd.DataFrame({"Field": ["Name", "Register number", "Programme", "Semester / Batch", "Date"],
                        "Details": ["Lekha Shree", "3122247001031", "M.Tech (Integrated) Computer Science & Engineering", "V / 2024-2029", datetime.date.today().strftime("%d-%m-%Y")]}), index=False, col_w=[4*cm, 12*cm]),
      Spacer(1, 8)]

S += [Paragraph("1. Aim", H1), P("To (i) evaluate matrix-based linear regression (closed form and gradient descent, with/without L2 regularization and standardization) for mobile-phone price prediction, and (ii) fit a single-layer (logistic) classifier for bank-note authentication, studying L2 regularization, a 3-D decision surface and the effect of outliers.")]

# ---------------- regression ----------------
n_tr, d_feat = Xtr.shape
S += [Paragraph("2. Part 1 – Regression: Mobile Phone Price Prediction", H1), Paragraph("2.1 Dataset and matrix formulation", H2),
      P(f"The dataset has {phones.shape[0]} phones and {d_feat} numeric predictors ({', '.join(FEATURES)}); the target is <b>{TARGET}</b>. Identifier columns were dropped and rows with missing values removed."),
      P(f"<b>Data matrix X</b> is {Xtr_b.shape[0]} × {Xtr_b.shape[1]} (training rows × (features + 1)); the leading column of ones lets the intercept be learned inside θ. <b>Label matrix y</b> is {ytr.shape[0]} × 1 (one real-valued price per phone, hence regression). <b>Parameter matrix θ</b> is {Xtr_b.shape[1]} × 1, fixed by the dimension of the data so that Xθ is conformable; each θⱼ is the change in predicted price for a one-unit change in feature j with the others fixed, so its sign and magnitude show the feature's influence on the prediction."),
      P(f"Only training rows are used to build X and y; the test rows ({len(Xte)} phones, 20%) are used only for evaluation, giving an unbiased estimate of generalization. Conditioning: cond(XᵀX) is {np.linalg.cond(Xtr_b.T @ Xtr_b):.2e} for raw features versus {np.linalg.cond(add_bias(sx_tmp.transform(Xtr)).T @ add_bias(sx_tmp.transform(Xtr))):.2e} after standardization, which is why gradient descent is run on standardized data."),
      Paragraph("2.2 Effect of the train/test split", H2), tbl(split_df.round(4), index=False)]
bi = split_df.test_R2_mean.idxmax(); hi = split_df.test_R2_std.idxmax()
S += [P(f"Over 30 random splits per ratio, mean test R² is highest at a test fraction of {split_df.test_size[bi]:.1f} ({split_df.test_R2_mean[bi]:.3f}), and the spread of test R² is largest at {split_df.test_size[hi]:.1f} (σ = {split_df.test_R2_std[hi]:.3f}). A small test set gives a noisy error estimate, while a large test set leaves less data for learning θ. The 80/20 split is a standard compromise and is used throughout."), *fig("reg_split_effect", "Figure 1: effect of the split ratio")]

q12 = res_q12
S += [Paragraph("2.3 Q1–Q3: closed-form vs gradient-descent solution", H2),
      P("Closed form: θ = (XᵀX)⁻¹Xᵀy (pseudo-inverse for numerical safety). Gradient descent: θ ← θ − α·(1/n)Xᵀ(Xθ − y) on standardized features and target, converted back to price units."),
      tbl(q12.round(4)),
      P(f"GD converged in {len(gd['hist'])} iterations; the largest difference between the two methods' test predictions is {np.max(np.abs(gd['pred_te'] - pred_cf_te)):.4g}, i.e. both reach the same optimum of the convex least-squares loss. On the test set R² = {q12.loc['Closed form (test)','R2']:.3f}, RMSE = {q12.loc['Closed form (test)','RMSE']:,.0f} and MAE = {q12.loc['Closed form (test)','MAE']:,.0f}; training R² is {q12.loc['Closed form (train)','R2']:.3f}"
        + (f", so the train–test gap of {q12.loc['Closed form (train)','R2'] - q12.loc['Closed form (test)','R2']:.3f} indicates some over-fitting/variance." if q12.loc['Closed form (train)','R2'] - q12.loc['Closed form (test)','R2'] > 0.05 else ", so there is little over-fitting.")),
      *fig("reg_q3_pred_vs_actual", "Figure 2: predicted vs actual (test) – closed form and gradient descent", 14), *fig("reg_q3_residuals", "Figure 3: residual analysis (closed form)", 14), *fig("reg_gd_convergence", "Figure 4: gradient-descent convergence", 9)]
S.append(P("Error analysis: points close to the red y = ŷ line are accurate; the residual plots show whether errors are centred on zero and whether they grow with price (funnel shape) – a sign that a purely linear model only approximates the price structure (brand, launch date and interactions are not captured)."))

imp_best = res_q4.loc["Ridge closed (test)"]; ols = res_q4.loc["OLS closed (test)"]
chg = (ols["RMSE"] - imp_best["RMSE"]) / ols["RMSE"] * 100
S += [Paragraph("2.4 Q4: L2 (ridge) regularization", H2),
      P(f"Ridge solves θ = (XᵀX + λI′)⁻¹Xᵀy (intercept not penalized); the same penalty is added to the GD gradient. λ was chosen by 5-fold cross-validation on the training set only: <b>λ* = {LAM_BEST:.4g}</b>."),
      tbl(res_q4.round(4)),
      P(f"Ridge closed form and ridge GD agree (max test-prediction difference {np.max(np.abs(rid_cf['pred_te'] - rid_gd['pred_te'])):.2g}). Compared with OLS, ridge test RMSE changes from {ols['RMSE']:,.0f} to {imp_best['RMSE']:,.0f} ({chg:+.2f}% {'improvement' if chg > 0 else 'change'}), test R² from {ols['R2']:.3f} to {imp_best['R2']:.3f}. "
        + ("Shrinkage reduced variance and gave slightly better generalization." if chg > 0.2 else "The benefit is small because the features are only moderately collinear and n is much larger than d; regularization mainly stabilizes the weights.")),
      *fig("reg_lambda_cv", "Figure 5: cross-validated MSE vs λ", 8), *fig("reg_q4_ridge_pred_vs_actual", "Figure 6: ridge predicted vs actual (test)", 14)]

r5 = res_q5
S += [Paragraph("2.5 Q5: ridge with and without standardization", H2), tbl(r5.round(4)),
      P(f"With the same λ, ridge on raw features reaches test R² = {r5.iloc[0]['R2']:.3f} versus {r5.iloc[1]['R2']:.3f} when standardized. Without standardization the L2 penalty is unfair: features measured in large units (e.g. battery) need only small weights and are barely penalized, while small-scale features (e.g. CPU frequency) need large weights and are heavily shrunk. Gradient descent is affected even more: on raw data it used {len(raw_gd['hist'])} iterations and ended with R² = {r5.iloc[2]['R2']:.3f}, whereas on standardized data it converged in {len(rid_gd['hist'])} iterations with R² = {r5.iloc[3]['R2']:.3f}. Standardization (statistics from the training set only, to avoid leakage) therefore gives a fair penalty, better conditioning and much faster, reliable convergence."),
      *fig("reg_q5_raw_vs_std", "Figure 7: test R² vs λ, raw vs standardized features", 10)]

bl = lam_df.loc[lam_df.test_R2.idxmax()]
S += [Paragraph("2.6 Q6: effect of λ", H2), tbl(lam_df.round(4), index=False),
      P(f"Test R² is best at λ = {bl['lambda']:g} (R² = {bl['test_R2']:.3f}). Small λ ≈ OLS (low bias, high variance); a moderate λ gives the best bias–variance trade-off; for very large λ all weights shrink to ≈ 0 (‖θ‖ falls from {lam_df['||theta||'].iloc[0]:.3f} to {lam_df['||theta||'].iloc[-1]:.4f}) and every prediction collapses to the mean price (R² → {lam_df['test_R2'].iloc[-1]:.3f}) – clear under-fitting, visible as a flat horizontal line of predictions."),
      *fig("reg_q6_lambda_grid", "Figure 8: predicted vs actual (test) for different λ", 16)]

top = imp.head(5)
S += [Paragraph("2.7 Q7: feature importance from the L2 weights", H2), tbl(imp[["rank", "feature", "ridge_weight", "OLS_weight"]].round(4), index=False),
      P("Because features are standardized, each weight is the price change per 1 σ change in that feature and weights are directly comparable. The most influential features are " + "; ".join(f"<b>{r.feature}</b> ({'raises' if r.ridge_weight > 0 else 'lowers'} price, w = {r.ridge_weight:.3f})" for r in top.head(3).itertuples()) +
        f". Ridge shrinks weak or collinear features towards zero (smallest: {imp.feature.iloc[-1]}, w = {imp.ridge_weight.iloc[-1]:.3f}), so large surviving weights mark real predictors. Caveat: correlated features share weight, so a small weight does not always mean irrelevance."),
      *fig("reg_q7_feature_importance", "Figure 9: ridge weights and coefficient paths", 15.5), *fig("reg_corr_heatmap", "Figure 10: feature correlation matrix", 10)]

S += [Paragraph("2.8 Effectiveness of the matrix-based approach", H2),
      P(f"The normal-equation/matrix approach gives an interpretable, fast model (test R² ≈ {q12.loc['Closed form (test)','R2']:.2f}, MAPE ≈ {q12.loc['Closed form (test)','MAPE_%']:.1f}%). Strengths: exact one-step solution, transparent weights. Limitations: assumes linearity, is sensitive to outliers and multicollinearity, and cannot capture interactions or non-linear price effects; polynomial features or tree ensembles would be natural improvements.")]

# ---------------- classification ----------------
S += [PageBreak(), Paragraph("3. Part 2 – Linear Classification: Bank Note Authentication", H1), Paragraph("3.1 Model and data split", H2),
      P(f"The dataset has {bank.shape[0]} notes with four wavelet features ({', '.join(CLS_FEATURES)}) and a binary label (class 1 fraction {yc.mean():.3f}). The model is a <b>single-layer neural network (logistic regression)</b>, p(y=1|x) = σ(wᵀx + b), trained by gradient descent on binary cross-entropy with optional L2 penalty (λ/2n)‖w‖². It is suitable because the task is two-class, the features are numeric and the classes are close to linearly separable."),
      tbl(pd.DataFrame({"Set": ["Train", "Test"], "Samples": [len(Xc_tr), len(Xc_te)], "Class-1 fraction": [yc_tr.mean(), yc_te.mean()]}), index=False),
      P("A stratified 80/20 split keeps the class ratio equal in both sets; features are standardized with training statistics only."),
      *fig("clf_feature_hists", "Figure 11: feature distributions by class", 16)]

a0, a1 = tab.iloc[1]["accuracy"], tab.iloc[3]["accuracy"]
S += [Paragraph("3.2 Q2: with and without L2 regularization", H2), tbl(tab[["accuracy", "precision", "recall", "F1", "TN", "FP", "FN", "TP"]].round(4)),
      P(f"λ was chosen by 5-fold CV on the training set (λ* = {LAM_CLF:g}). Test accuracy is {a0:.4f} without L2 and {a1:.4f} with L2; the weight norm falls from {np.linalg.norm(w0[1:]):.3f} to {np.linalg.norm(w1[1:]):.3f}. "
        + ("L2 regularization gave a small gain in test accuracy while keeping the weights smaller." if a1 > a0 + 1e-9 else "L2 regularization gave essentially the same accuracy (the data are almost linearly separable) but with smaller, more stable weights; without a penalty the weights keep growing to push probabilities towards 0/1.")),
      *fig("clf_loss_curves", "Figure 12: training loss", 8)]

bi = int(np.argmax(te_a))
S += [Paragraph("3.3 Q3: training and test accuracy vs λ", H2), *fig("clf_acc_vs_lambda", "Figure 13: accuracy vs λ", 11),
      P(f"Accuracy is flat for small λ, reaches its best test value of {max(te_a):.4f} at λ ≈ {lams[bi]:.3g}, and falls to {te_a[-1]:.3f} at λ = {lams[-1]:g} as the weights are over-shrunk (under-fitting). Train and test accuracy stay close together, so over-fitting is mild for this low-dimensional problem.")]

S += [Paragraph("3.4 Q4: 3-D visualization of the three most important features", H2),
      P(f"Ranked by |w| in the standardized L2 model: " + ", ".join(f"{k} ({v:.3f})" for k, v in imp_c.items()) + f". The top three – <b>{', '.join(TOP3)}</b> – were used for the 3-D plot. A classifier restricted to these three features still reaches {acc3:.4f} test accuracy; the grey surface is its decision plane wᵀx + b = 0 separating the two classes."),
      *fig("clf_q4_3d", "Figure 14: test data in 3-D with the learned decision plane", 16)]

S += [Paragraph("3.5 Q5: introducing outliers", H2),
      P(f"A fraction of 5% of the training points ({int(out_mask.sum())} of {len(Xc_tr)}) was shifted by ±6 standard deviations along {', '.join(TOP3)} (random sign, labels unchanged). The standardizer was re-fitted on the contaminated training data, and models were tested on the clean test set."),
      *fig("clf_q5_outliers_3d", "Figure 15: training data before and after shifting points", 15)]

S += [Paragraph("3.6 Q6: classifier on outlier-injected data", H2), tbl(cmp_df.round(4), index=False),
      P(f"Without L2 the clean-test accuracy changes from {cmp_df.clean_test_acc.iloc[0]:.4f} to {cmp_df.clean_test_acc.iloc[2]:.4f} (drop {drop0:.4f}); with L2 from {cmp_df.clean_test_acc.iloc[1]:.4f} to {cmp_df.clean_test_acc.iloc[3]:.4f} (drop {dropL:.4f}). The cosine similarity between the clean and contaminated weight vectors is {cos_sim:.3f}. "
        + ("The shifted points clearly pull the decision boundary and change the weights, and the outliers also inflate the mean and σ used for standardization. " if max(drop0, dropL) > 0.01 else "At this contamination level the boundary moves only slightly, so the classifier is fairly robust. ")
        + ("L2 regularization reduced the damage by limiting weight growth." if dropL < drop0 - 1e-9 else "L2 regularization did not reduce the damage here: it limits weight growth but cannot stop extreme points from tilting the boundary.")
        + " Mitigations: outlier detection/removal, robust scaling (median/IQR), winsorizing or a robust loss."),
      *fig("clf_q6_outlier_sweep", "Figure 16: clean-test accuracy vs % of shifted training points", 10)]

S += [Paragraph("4. Conclusion", H1), *bullets([
      f"Matrix-based linear regression is an interpretable, fast baseline (test R² ≈ {q12.loc['Closed form (test)','R2']:.2f}); closed form and gradient descent give identical predictions once features are standardized.",
      f"L2 regularization (λ* = {LAM_BEST:.3g}) and standardization make the weights stable and comparable; the most influential features are {', '.join(top.feature.head(3))}.",
      f"Logistic regression separates genuine and forged notes with {a1*100:.1f}% test accuracy; L2 controls weight growth and very large λ under-fits.",
      f"Shifting 5% of training points changed clean-test accuracy by {drop0*100:.1f} points (no L2) / {dropL*100:.1f} points (L2): outliers distort the boundary and need to be handled explicitly."])]

REPORT = "CAT1_Report_Lekha_Shree_3122247001031.pdf"
doc = SimpleDocTemplate(REPORT, pagesize=A4, leftMargin=2*cm, rightMargin=2*cm, topMargin=1.8*cm, bottomMargin=1.8*cm,
                        title="CAT Assignment 1 – ICS1502", author="Lekha Shree (3122247001031)")
doc.build(S, onFirstPage=lambda c, d_: None, onLaterPages=lambda c, d_: (c.setFont("DV", 8), c.drawString(2*cm, 1*cm, "Lekha Shree – 3122247001031 – ICS1502 CAT Assignment 1"), c.drawRightString(A4[0] - 2*cm, 1*cm, f"Page {c.getPageNumber()}")))
print("Report written to", REPORT)